In [1]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, confusion_matrix

# ---------------- USER SETTINGS ----------------
DATA_PATH = "data/corn_2016_2023.parquet"   # change crop file here
TARGET_COL = "yield"                       # regression target
GROUP_COL  = "farm_name"                   # group split key
N_SPLITS   = 5

# feature choices
INCLUDE_YEAR_LAT = True                    # you said: include Year + latitude
USE_LONGITUDE = False                      # you decided to drop longitude
# ------------------------------------------------

# Load
df = pd.read_parquet(DATA_PATH)

# Drop rows with missing target
df = df[df[TARGET_COL].notna()].copy()

# Clean: drop old wrong normalized cols if present
bad_norm = [c for c in ["normalized_yield_pct", "normalized_yield_z"] if c in df.columns]

# Clean: remove SAR duplicates ending with _2
sar_dup2 = [c for c in df.columns if (c.startswith("VV_") or c.startswith("VH_")) and c.endswith("_2")]

df = df.drop(columns=bad_norm + sar_dup2, errors="ignore")

print("Rows:", len(df), "Cols:", len(df.columns))
print("Dropped:", bad_norm, "and", len(sar_dup2), "SAR _2 cols")

# Continuous target (float)
y = df[TARGET_COL].to_numpy(dtype=np.float32)

# Groups
groups = df[GROUP_COL].astype(str).to_numpy()
print("Unique groups:", len(np.unique(groups)))

# CV folds (fixed across models)
gkf = GroupKFold(n_splits=N_SPLITS)
folds = list(gkf.split(np.zeros(len(df)), y, groups=groups))
print("Built folds:", len(folds))

# ------------ Metrics helpers ------------
def regression_metrics(y_true, y_pred):
    """Return R², RMSE, MAE."""
    r2 = r2_score(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    return r2, rmse, mae

def zone_thresholds_from_train(y_train, q_low=0.4, q_high=0.6):
    """Compute yield thresholds from TRAIN only (prevents leakage)."""
    low = np.quantile(y_train, q_low)
    high = np.quantile(y_train, q_high)
    return low, high

def yield_to_zone(y_vals, low, high):
    """Convert continuous yield to zones using thresholds."""
    # 0=low, 1=medium, 2=high
    z = np.zeros_like(y_vals, dtype=np.int64)
    z[(y_vals >= low) & (y_vals <= high)] = 1
    z[y_vals > high] = 2
    return z

def zone_metrics(y_true, y_pred, y_train_for_thresholds):
    """
    Evaluate 'zone accuracy' + confusion matrix for regression predictions.
    Thresholds are computed from TRAIN yields only.
    """
    low, high = zone_thresholds_from_train(y_train_for_thresholds, 0.4, 0.6)
    z_true = yield_to_zone(y_true, low, high)
    z_pred = yield_to_zone(y_pred, low, high)
    acc = (z_true == z_pred).mean()
    cm = confusion_matrix(z_true, z_pred, labels=[0,1,2])
    return float(acc), cm

def summarize_cv_regression(r2s, rmses, maes, name="MODEL"):
    print(f"\n{name} CV Summary (regression)")
    print(f"R²:   {np.mean(r2s):.3f} ± {np.std(r2s):.3f}")
    print(f"RMSE: {np.mean(rmses):.3f} ± {np.std(rmses):.3f}")
    print(f"MAE:  {np.mean(maes):.3f} ± {np.std(maes):.3f}")

def summarize_cv_zones(zone_accs, cm_sum, name="MODEL"):
    print(f"\n{name} CV Summary (zone-from-regression)")
    print(f"Zone Accuracy: {np.mean(zone_accs):.3f} ± {np.std(zone_accs):.3f}")
    print("Zone confusion matrix sum (rows=true low/med/high, cols=pred):\n", cm_sum)





def build_year_mean_map(years_train, y_train_raw):
    """TRAIN-only map: year -> mean yield in that year."""
    year_means = {}
    for yr in np.unique(years_train):
        vals = y_train_raw[years_train == yr]
        year_means[int(yr)] = float(np.mean(vals))
    return year_means

def lookup_year_means(years, year_means, fallback):
    """Per-sample mean yield by year using TRAIN-derived map."""
    out = np.empty(len(years), dtype=np.float32)
    for i, yr in enumerate(years):
        out[i] = year_means.get(int(yr), fallback)
    return out

def zone_metrics_pm10_from_ratio(y_true_ratio, y_pred_ratio):
    """Zones in ratio space: low<0.9, med 0.9-1.1, high>1.1."""
    z_true = np.where(y_true_ratio < 0.9, 0, np.where(y_true_ratio <= 1.1, 1, 2))
    z_pred = np.where(y_pred_ratio < 0.9, 0, np.where(y_pred_ratio <= 1.1, 1, 2))
    acc = float((z_true == z_pred).mean())
    cm = confusion_matrix(z_true, z_pred, labels=[0,1,2])
    return acc, cm

Rows: 2108996 Cols: 349
Dropped: ['normalized_yield_pct', 'normalized_yield_z'] and 20 SAR _2 cols
Unique groups: 251
Built folds: 5


# Build SAR sequence tensor (needed for RNN/LSTM/Transformer)
## What / Why

Convert VV_#### and VH_#### columns into a time series: (N, T, 2).

Preprocessing

Drop _2 columns (already done in shared setup)

Align VV and VH by suffix

Replace NaN/inf and standardize per fold (done later)

In [2]:
import re
import numpy as np

# ---------- BUILD SAR SEQUENCE (X_seq) ----------
def suffix(col):
    m = re.match(r"^(VV|VH)_(\d+)$", col)
    return m.group(2) if m else None

vv_cols = [c for c in df.columns if c.startswith("VV_")]
vh_cols = [c for c in df.columns if c.startswith("VH_")]

vv_map = {suffix(c): c for c in vv_cols if suffix(c) is not None}
vh_map = {suffix(c): c for c in vh_cols if suffix(c) is not None}

common_suffixes = sorted(set(vv_map.keys()) & set(vh_map.keys()))
T = len(common_suffixes)

print("Time steps (T):", T)

vv_sorted = [vv_map[s] for s in common_suffixes]
vh_sorted = [vh_map[s] for s in common_suffixes]

X_seq = np.zeros((len(df), T, 2), dtype=np.float32)
X_seq[:, :, 0] = df[vv_sorted].to_numpy(dtype=np.float32)
X_seq[:, :, 1] = df[vh_sorted].to_numpy(dtype=np.float32)

print("X_seq shape:", X_seq.shape)
print("NaNs in X_seq:", np.isnan(X_seq).sum())

# ---------- STATIC FEATURES: Year + latitude ----------
STATIC_COLS = ["Year", "latitude"]  # no longitude
X_static = df[STATIC_COLS].to_numpy(dtype=np.float32)
print("X_static shape:", X_static.shape)


Time steps (T): 152
X_seq shape: (2108996, 152, 2)
NaNs in X_seq: 536282261
X_static shape: (2108996, 2)


# Transformer (SAR sequence only)

Same preprocessing + fold standardization, model:

In [3]:
import time
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# ----------------- GPU SETTINGS -----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    torch.set_num_threads(50)
    torch.set_num_interop_threads(4)

# ----------------- HELPERS -----------------
def fmt_time(seconds: float) -> str:
    seconds = max(0.0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:d}h {m:02d}m {s:02d}s" if h > 0 else f"{m:02d}m {s:02d}s"

def preprocess_sar_fold(X_tr, X_te, add_mask=True):
    mask_tr = np.isnan(X_tr).astype(np.float32)
    mask_te = np.isnan(X_te).astype(np.float32)

    X_tr_filled = np.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)
    X_te_filled = np.nan_to_num(X_te, nan=0.0, posinf=0.0, neginf=0.0)

    mean = X_tr_filled.reshape(-1, X_tr.shape[-1]).mean(axis=0)
    std  = X_tr_filled.reshape(-1, X_tr.shape[-1]).std(axis=0) + 1e-6

    X_tr_std = (X_tr_filled - mean) / std
    X_te_std = (X_te_filled - mean) / std

    if add_mask:
        X_tr_out = np.concatenate([X_tr_std, mask_tr], axis=2)
        X_te_out = np.concatenate([X_te_std, mask_te], axis=2)
    else:
        X_tr_out, X_te_out = X_tr_std, X_te_std

    return X_tr_out.astype(np.float32), X_te_out.astype(np.float32)

def standardize_static_fold(S_tr, S_te):
    mu = S_tr.mean(axis=0)
    sd = S_tr.std(axis=0) + 1e-6
    return ((S_tr - mu) / sd).astype(np.float32), ((S_te - mu) / sd).astype(np.float32)

# ----------------- DATASET -----------------
class SeqStaticDataset(Dataset):
    def __init__(self, Xseq, Xstat, y):
        self.Xseq  = torch.as_tensor(Xseq, dtype=torch.float32)
        self.Xstat = torch.as_tensor(Xstat, dtype=torch.float32)
        self.y     = torch.as_tensor(y, dtype=torch.float32)  # regression target

    def __len__(self): 
        return self.y.shape[0]

    def __getitem__(self, i):
        return self.Xseq[i], self.Xstat[i], self.y[i]

# ----------------- MODEL -----------------
class TimeSeriesTransformerRegressor(nn.Module):
    """
    Transformer encoder over time with learned positional embedding.
    Input: (B, T, C) where C=4 (VV,VH,maskVV,maskVH)
    Fuses static (Year, latitude).
    Output: continuous yield.
    """
    def __init__(self, input_dim=4, d_model=128, nhead=8, num_layers=4,
                 dropout=0.1, max_len=512, stat_dim=2):
        super().__init__()

        self.proj = nn.Linear(input_dim, d_model)

        # learned positional embedding (simple + works well)
        self.pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        nn.init.normal_(self.pos, std=0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dropout=dropout,
            batch_first=True,
            activation="gelu",
            norm_first=True
        )
        self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # static fusion
        self.stat_mlp = nn.Sequential(
            nn.Linear(stat_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU()
        )

        self.head = nn.Sequential(
            nn.LayerNorm(d_model + 32),
            nn.Linear(d_model + 32, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 1)
        )

    def forward(self, x_seq, x_stat):
        B, T, C = x_seq.shape
        z = self.proj(x_seq) + self.pos[:, :T, :]
        z = self.enc(z)
        z = z.mean(dim=1)  # mean pool over time

        s = self.stat_mlp(x_stat)
        h = torch.cat([z, s], dim=1)
        return self.head(h).squeeze(1)

@torch.no_grad()
def eval_regression(model, loader):
    model.eval()
    yt, yp = [], []
    for xseq, xstat, yb in loader:
        xseq = xseq.to(device, non_blocking=True)
        xstat = xstat.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        pred = model(xseq, xstat)
        yt.append(yb.detach().cpu().numpy())
        yp.append(pred.detach().cpu().numpy())
    return np.concatenate(yt), np.concatenate(yp)

# ----------------- TRAIN CV -----------------
BATCH_SIZE = 4096   # Keep at 4096; Transformers use more VRAM than LSTMs
EPOCHS = 20         # Transformers need more time to converge on patterns
LR = 4e-4           # Slightly lower LR is often more stable for Transformers
PIN = (device.type == "cuda")
use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

r2s, rmses, maes = [], [], []
zone_accs = []
cm_zone_sum = np.zeros((3, 3), dtype=int)

overall_start = time.time()
total_folds = len(folds)

for k, (tr, te) in enumerate(folds, start=1):
    print(f"\n===== Fold {k}/{total_folds} (Transformer regression) =====")
    fold_start = time.time()

    # 1. Preprocessing
    X_tr_seq, X_te_seq = preprocess_sar_fold(X_seq[tr], X_seq[te], add_mask=True)
    S_tr, S_te = standardize_static_fold(X_static[tr], X_static[te])

    # 2. YEAR-NORMALIZATION LOGIC
    years_train = df["Year"].to_numpy()[tr]
    years_test  = df["Year"].to_numpy()[te]
    y_raw_tr = y[tr]
    y_raw_te = y[te]

    year_means_map = build_year_mean_map(years_train, y_raw_tr)
    global_fallback = float(np.mean(y_raw_tr))

    mu_tr = lookup_year_means(years_train, year_means_map, global_fallback)
    mu_te = lookup_year_means(years_test,  year_means_map, global_fallback)

    # Normalize targets: predictable ratio for the Transformer
    y_tr_norm = (y_raw_tr / mu_tr).astype(np.float32)
    y_te_norm = (y_raw_te / mu_te).astype(np.float32) 

    # 3. DataLoaders (using normalized targets)
    dl_tr = DataLoader(SeqStaticDataset(X_tr_seq, S_tr, y_tr_norm),
                       batch_size=BATCH_SIZE, shuffle=True, pin_memory=PIN)
    dl_te = DataLoader(SeqStaticDataset(X_te_seq, S_te, y_te_norm),
                       batch_size=BATCH_SIZE, shuffle=False, pin_memory=PIN)

    # 4. Model Setup
    model = TimeSeriesTransformerRegressor(
        input_dim=4, d_model=128, nhead=8, num_layers=4,
        max_len=max(512, X_tr_seq.shape[1] + 1), stat_dim=2
    ).to(device)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    loss_fn = nn.SmoothL1Loss(beta=1.0)

    # 5. Training Loop
    for ep in range(1, EPOCHS + 1):
        epoch_start = time.time()
        model.train()
        for xseq, xstat, yb in dl_tr:
            xseq, xstat, yb = xseq.to(device), xstat.to(device), yb.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=use_amp):
                pred = model(xseq, xstat)
                loss = loss_fn(pred, yb)
            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

        # Validate (on normalized scale)
        y_true_norm, y_pred_norm = eval_regression(model, dl_te)
        r2_norm, _, _ = regression_metrics(y_true_norm, y_pred_norm)
        print(f"  epoch {ep:02d} | R²(norm)={r2_norm:.3f} | time={fmt_time(time.time()-epoch_start)}", end="\r")

    # 6. EVALUATION: Convert back to Raw Yield for Metrics
    y_true_norm, y_pred_norm = eval_regression(model, dl_te)
    
    pred_raw = y_pred_norm * mu_te # Inverse transform
    true_raw = y_raw_te 

    r2, rmse, mae = regression_metrics(true_raw, pred_raw)
    zacc, cmz = zone_metrics_pm10_from_ratio(y_true_norm, y_pred_norm)

    r2s.append(r2); rmses.append(rmse); maes.append(mae)
    zone_accs.append(zacc); cm_zone_sum += cmz

    print(f"\nFold {k} FINAL: R²={r2:.3f} | RMSE={rmse:.3f} | Zone Acc={zacc:.3f}")
    
summarize_cv_regression(r2s, rmses, maes, name="TimeSeriesTransformerRegressor (SAR+Year+lat, GPU+AMP)")
summarize_cv_zones(zone_accs, cm_zone_sum, name="TimeSeriesTransformerRegressor (zone eval)")
print("Total wall time:", fmt_time(time.time() - overall_start))

Device: cuda
GPU: NVIDIA H200

===== Fold 1/5 (Transformer regression) =====


/tmp/ipykernel_189420/4014418182.py:91: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


  epoch 20 | R²(norm)=0.291 | time=01m 18s
Fold 1 FINAL: R²=0.356 | RMSE=37.650 | Zone Acc=0.534

===== Fold 2/5 (Transformer regression) =====


/tmp/ipykernel_189420/4014418182.py:91: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


  epoch 20 | R²(norm)=0.369 | time=01m 12s
Fold 2 FINAL: R²=0.382 | RMSE=37.682 | Zone Acc=0.569

===== Fold 3/5 (Transformer regression) =====


...

  epoch 20 | R²(norm)=0.223 | time=01m 12s
Fold 3 FINAL: R²=0.269 | RMSE=40.795 | Zone Acc=0.479

===== Fold 4/5 (Transformer regression) =====


/tmp/ipykernel_189420/4014418182.py:91: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


  epoch 20 | R²(norm)=0.137 | time=01m 12s
Fold 4 FINAL: R²=0.167 | RMSE=44.293 | Zone Acc=0.561

===== Fold 5/5 (Transformer regression) =====


/tmp/ipykernel_189420/4014418182.py:91: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.enc = nn.TransformerEncoder(enc_layer, num_layers=num_layers)


  epoch 20 | R²(norm)=0.131 | time=01m 18s
Fold 5 FINAL: R²=0.257 | RMSE=37.864 | Zone Acc=0.500

TimeSeriesTransformerRegressor (SAR+Year+lat, GPU+AMP) CV Summary (regression)
R²:   0.286 ± 0.077
RMSE: 39.657 ± 2.605
MAE:  29.451 ± 1.537

TimeSeriesTransformerRegressor (zone eval) CV Summary (zone-from-regression)
Zone Accuracy: 0.529 ± 0.035
Zone confusion matrix sum (rows=true low/med/high, cols=pred):
 [[415348 221212  34962]
 [152188 319015 110281]
 [ 66407 408834 380749]]
Total wall time: 2h 09m 08s
